In [1]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!pip install -e ".[dev,vlm,gen]" -q

Cloning into 'TIGeR-Text-Image-Generative-Repair'...
remote: Enumerating objects: 744, done.
remote: Counting objects: 100% (744/744), done.
remote: Compressing objects: 100% (524/524), done.
remote: Total 744 (delta 290), reused 656 (delta 203), pack-reused 0 (from 0)
Receiving objects: 100% (744/744), 3.36 MiB | 26.64 MiB/s, done.
Resolving deltas: 100% (290/290), done.
/kaggle/working/TIGeR-Text-Image-Generative-Repair
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tiger (pyproject.toml) ... done


In [2]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("gemini api")
    os.environ["GEMINI_API_KEY"] = gemini_key
    print("✅ Gemini API Key loaded successfully from Kaggle Secrets!")
except Exception as e:
    print("❌ Failed to load Gemini API Key. Did you add it to Kaggle Secrets? Error:", e)

✅ Gemini API Key loaded successfully from Kaggle Secrets!


## 1. Import ABO Dataset
Make sure you have added the `khyeh0719/amazon-berkeley-objects-small` dataset to this notebook.

In [ ]:
!python -m tiger.cli import-abo \
    --listings-dir /kaggle/input/amazon-berkeley-objects-small/listings/metadata \
    --images-csv /kaggle/input/amazon-berkeley-objects-small/images/metadata/images.csv.gz \
    --images-dir /kaggle/input/amazon-berkeley-objects-small/images/small

## 2. Calibrate on ABO
This fits the new similarity thresholds for the non-fashion domain.

In [ ]:
!python -m tiger.cli calibrate

## 3. Retrain Arbiter
This retrains the Logistic Regression router on the new ABO-domain noise patterns.

In [ ]:
!python -m tiger.cli train-arbiter

## 4. Run Repair Ablation
This evaluates the repair pipeline using the Independent Verifier (SigLIP) and Generative Fallback.

In [ ]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

## 5. View Results
Compare this table with the Fashion results in your paper.

In [ ]:
import pandas as pd
df = pd.read_csv("data/outputs/repair_ablations_summary_run1.csv")
print(df)